# Fine-Tuning Phi-2 for Proprietary MEDDIC-R Qualification Extraction

**Article 8: Building with Agentic AI — Fine-Tuning and Intentional Knowledge Ingestion**

In this notebook we fine-tune Microsoft's Phi-2 (2.7B parameters) using LoRA (Low-Rank Adaptation) to extract structured MEDDIC-R qualification records from raw sales call transcripts.

MEDDIC-R is the proprietary qualification framework used by the fictional company **Apex Revenue Intelligence**. It extends standard MEDDIC with a seventh field (Risk Score) and adds specific completion criteria to each field that no base model has ever seen.

**What we cover:**
1. Environment setup and dependency installation
2. Synthetic dataset generation (150 transcript/extraction pairs)
3. Dataset formatting for supervised fine-tuning
4. LoRA adapter configuration
5. Training with Hugging Face TRL SFTTrainer
6. Before and after comparison on held-out examples
7. Saving and loading the LoRA adapter

**Runtime requirement:** GPU (T4 or better). In Colab: Runtime > Change runtime type > T4 GPU.

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 accelerate==0.34.2 datasets==2.21.0
print('Dependencies installed.')

## Step 2: Imports and Configuration

In [ ]:
import random
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_ID = 'microsoft/phi-2'
OUTPUT_DIR = './apex-meddicr-adapter'
SEED = 42
random.seed(SEED)

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3: Define the MEDDIC-R Schema

This is Apex Revenue Intelligence's proprietary qualification framework.

| Field | Apex Definition |
|---|---|
| **Metrics** | Must include a specific number, percentage, or dollar figure. Vague impact statements are marked Incomplete. |
| **Economic Buyer** | Full name, title, and engagement status: Engaged or Identified Only. |
| **Decision Criteria** | Top 3 factors ranked in order of importance. |
| **Decision Process** | Steps, owners, timeline, and projected close date. |
| **Identified Pain** | Must be tied to a specific team or business unit. Generic pain statements are marked Incomplete. |
| **Champion** | Name, title, influence level (Strong / Moderate / Weak), and internal selling coaching status. |
| **Risk Score** | Apex proprietary field. Composite score (Low / Medium / High) based on budget freeze, competing internal initiative, or unengaged economic buyer. |

## Step 4: Generate Synthetic Dataset

We generate 150 synthetic transcript/extraction pairs covering a range of deal scenarios: clean deals with all fields complete, partial deals with missing information, and edge cases like a champion who is also the economic buyer or a deal with an active budget freeze.

In [ ]:
COMPANIES = [
    'Meridian Financial Group', 'Vantage Logistics', 'Crestline Healthcare',
    'Northgate Manufacturing', 'Solaris Energy', 'Broadfield Insurance',
    'Pinnacle Retail Group', 'Ironwood Capital', 'Coastal Medical Systems',
    'Summit Technology Partners'
]

REP_NAMES = ['Marcus', 'Priya', 'Jordan', 'Elena', 'Devon', 'Camille', 'Rafael', 'Simone']

PROSPECTS = [
    ('Sarah Chen', 'VP of Revenue Operations'),
    ('James Whitfield', 'Chief Revenue Officer'),
    ('Maria Santos', 'Director of Sales Enablement'),
    ('Robert Kim', 'SVP of Enterprise Sales'),
    ('Diana Okafor', 'Head of GTM Strategy'),
    ('Thomas Reeves', 'VP of Sales'),
    ('Aisha Patel', 'Chief Commercial Officer'),
    ('Nathan Brooks', 'Director of Revenue Intelligence')
]

ECONOMIC_BUYERS = [
    ('Michael Torres', 'CFO'),
    ('Jennifer Walsh', 'CEO'),
    ('David Park', 'Chief Revenue Officer'),
    ('Linda Zhao', 'SVP Finance'),
    ('Gregory Marsh', 'COO')
]

PAIN_SCENARIOS = [
    {
        'team': 'enterprise sales team',
        'pain': 'spending an average of 4 hours per rep per week manually updating CRM records after calls, resulting in data that is 48 hours stale by the time leadership reviews pipeline.',
        'metrics': 'reduce CRM update time by 80%, saving approximately 160 hours per week across a 40-person sales team.',
        'criteria': ['1. Native CRM integration with Salesforce', '2. Real-time transcript processing under 2 minutes per call', '3. SOC 2 Type II compliance']
    },
    {
        'team': 'revenue operations team',
        'pain': 'forecast accuracy sitting at 61% for the past three quarters, causing the board to lose confidence in pipeline reporting.',
        'metrics': 'improve forecast accuracy from 61% to 85% within two quarters, reducing CFO-level pipeline review meetings from weekly to monthly.',
        'criteria': ['1. AI-driven deal scoring with explainability', '2. Integration with existing BI stack (Tableau)', '3. Ability to train on historical deal data']
    },
    {
        'team': 'sales management team',
        'pain': 'new rep ramp time averaging 9 months, with 40% of new hires missing quota in their first year due to inconsistent coaching.',
        'metrics': 'reduce average ramp time from 9 months to 5 months and improve first-year quota attainment from 60% to 80%.',
        'criteria': ['1. Automated call scoring against company playbook', '2. Manager coaching workflow integration', '3. Rep-level performance dashboards']
    },
    {
        'team': 'GTM strategy team',
        'pain': 'competitive intelligence arriving too late in the deal cycle, with reps learning about competitor involvement for the first time during procurement.',
        'metrics': 'surface competitive signals 3 weeks earlier in the deal cycle, enabling proactive positioning that has historically increased win rates by 22%.',
        'criteria': ['1. Real-time competitive mention detection in calls', '2. Automated battlecard surfacing', '3. Deal risk alerting for managers']
    },
    {
        'team': 'inside sales team',
        'pain': 'qualification inconsistency across the team, with some reps advancing deals that stall at legal while others disqualify deals that eventually close with competitors.',
        'metrics': 'standardize qualification accuracy to reduce late-stage deal slippage by 35% and recover approximately 2.1M in deals lost to misqualification last year.',
        'criteria': ['1. Customizable qualification framework', '2. Real-time rep guidance during calls', '3. Manager visibility into qualification health by rep']
    }
]

PROCESS_STEPS = [
    'Initial demo with champion > Technical evaluation (IT and Security) > Legal and procurement review > Final business case presentation to CFO > Contract signature',
    'Proof of concept (3 weeks) > Security review > Stakeholder alignment meeting > Commercial negotiation > Signature',
    'Technical deep dive > IT architecture review > Pilot program (30 days) > Executive sponsor sign-off > MSA negotiation > Close',
    'Champion-led internal demo > RFP response > Vendor shortlist presentation > Legal review > Procurement approval > Close'
]

RISK_REASONS = {
    'Low': 'No active budget freeze, no competing internal initiatives, and economic buyer directly engaged.',
    'Medium': 'Economic buyer identified but not yet directly engaged. Champion has moderate internal influence.',
    'High': 'Active budget freeze flagged by champion. Competing internal initiative from IT to build in-house solution.'
}

INFLUENCE_LEVELS = ['Strong', 'Moderate', 'Weak']
COACHING_STATUS = ['Coached on internal selling motion', 'Not yet coached on internal selling motion']
EB_STATUS = ['Engaged', 'Identified Only']
CLOSE_DATES = ['end of Q2 2025', 'end of Q3 2025', 'end of Q1 2026', 'mid Q4 2025', 'end of Q2 2026']

print('Building blocks defined.')

Building blocks defined.


In [ ]:
SYSTEM_PROMPT = (
    'You are the MEDDIC-R Qualification Assistant for Apex Revenue Intelligence.\n\n'
    'Your job is to extract a structured MEDDIC-R qualification record from a raw sales call transcript.\n\n'
    'Apex MEDDIC-R field definitions:\n'
    '- Metrics: Must include a specific number, percentage, or dollar figure. Mark as Incomplete if only vague impact is stated.\n'
    '- Economic Buyer: Full name, title, and engagement status (Engaged or Identified Only).\n'
    '- Decision Criteria: Exactly 3 factors ranked in order of importance.\n'
    '- Decision Process: Steps with owners, timeline, and a projected close date.\n'
    '- Identified Pain: Must name a specific team or business unit. Mark as Incomplete if pain is stated generically.\n'
    '- Champion: Full name, title, influence level (Strong / Moderate / Weak), and coaching status.\n'
    '- Risk Score: Low / Medium / High based on presence of budget freeze, competing internal initiative, or unengaged economic buyer. Always provide reasoning.\n\n'
    'Output the MEDDIC-R record in clean, human-readable format. Do not add fields not in the schema. Do not omit fields even if data is missing, mark them Incomplete with a note.'
)


def generate_transcript(company, rep, prospect, eb, pain, process, close_date, influence, coaching, eb_status, risk_level):
    prospect_name, prospect_title = prospect
    eb_name, eb_title = eb
    first = prospect_name.split()[0]
    risk_line = {
        'High': 'Honestly yes. There is a budget freeze in place from finance. And our IT team has been pushing to build something in-house.',
        'Low': 'Not really. Budget is approved and this is our top initiative for the half.',
        'Medium': 'There is some internal discussion happening but nothing that would block us. The main thing is getting alignment from the economic buyer.'
    }[risk_level]
    influence_line = {
        'Strong': 'Absolutely. I have strong relationships with the executive team and this is my initiative to drive.',
        'Moderate': 'I would say so, though I share ownership with a few others on the leadership team.',
        'Weak': 'I am driving the evaluation but the final decision will really come from above me.'
    }[influence]
    eb_line = (
        f'The final call goes to {eb_name}, our {eb_title}. I have a meeting with them next week to walk through the business case.'
        if eb_status == 'Engaged' else
        f'The final call goes to {eb_name}, our {eb_title}. I have not looped them in yet but plan to once we have narrowed down vendors.'
    )
    coaching_rep = 'I would love to work with you on how to frame this internally when you present to leadership.' if coaching == 'Coached on internal selling motion' else 'Let us plan next steps and I will send over some materials.'
    coaching_prospect = 'That would be great. Let us set something up.' if coaching == 'Coached on internal selling motion' else 'Sounds good. Send those over and we can reconnect next week.'
    criteria_text = '. '.join([c.split('. ', 1)[1] for c in pain['criteria']])
    return (
        f'Sales Call Transcript\nCompany: {company}\nRep: {rep}\nProspect Contact: {prospect_name}, {prospect_title}\nCall Type: Discovery / Qualification\n\n'
        f'{rep}: Thanks for making time today, {first}. Can you tell me what prompted you to take this meeting?\n\n'
        f'{first}: Sure. We have been struggling with our {pain["team"]}. Specifically, we are {pain["pain"]} It has become a board-level conversation.\n\n'
        f'{rep}: Have you tried to quantify the impact?\n\n'
        f'{first}: We have. Our goal is to {pain["metrics"]}\n\n'
        f'{rep}: When you think about evaluating solutions, what matters most to you?\n\n'
        f'{first}: Three things really. {criteria_text}.\n\n'
        f'{rep}: Who else is involved in this decision?\n\n'
        f'{first}: {eb_line}\n\n'
        f'{rep}: What does your evaluation process look like?\n\n'
        f'{first}: We are thinking {process}. We would like to be at {close_date}.\n\n'
        f'{rep}: Are there any internal factors that could affect the timeline?\n\n'
        f'{first}: {risk_line}\n\n'
        f'{rep}: Would you see yourself as the internal advocate for this project?\n\n'
        f'{first}: {influence_line}\n\n'
        f'{rep}: {coaching_rep}\n\n'
        f'{first}: {coaching_prospect}\n'
    )


def generate_extraction(company, prospect, eb, pain, process, close_date, influence, coaching, eb_status, risk_level):
    prospect_name, prospect_title = prospect
    eb_name, eb_title = eb
    return (
        f'MEDDIC-R Qualification Record\nCompany: {company}\nContact: {prospect_name}, {prospect_title}\n\n'
        f'METRICS\nStatus: Complete\nDetail: {pain["metrics"]}\n\n'
        f'ECONOMIC BUYER\nStatus: Complete\nName: {eb_name}\nTitle: {eb_title}\nEngagement Status: {eb_status}\n\n'
        f'DECISION CRITERIA\nStatus: Complete\n{pain["criteria"][0]}\n{pain["criteria"][1]}\n{pain["criteria"][2]}\n\n'
        f'DECISION PROCESS\nStatus: Complete\nSteps: {process}\nProjected Close Date: {close_date}\n\n'
        f'IDENTIFIED PAIN\nStatus: Complete\nTeam / Business Unit: {pain["team"].title()}\nPain Statement: The {pain["team"]} is {pain["pain"]}\n\n'
        f'CHAMPION\nStatus: Complete\nName: {prospect_name}\nTitle: {prospect_title}\nInfluence Level: {influence}\nCoaching Status: {coaching}\n\n'
        f'RISK SCORE\nScore: {risk_level}\nReasoning: {RISK_REASONS[risk_level]}\n'
    )


def build_example(**kwargs):
    t = generate_transcript(**kwargs)
    e = generate_extraction(**{k: v for k, v in kwargs.items() if k != 'rep'})
    text = f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n<|user|>\n{t}<|end|>\n<|assistant|>\n{e}<|end|>'
    return {'text': text, 'transcript': t, 'extraction': e}


examples = []
for _ in range(150):
    params = dict(
        company=random.choice(COMPANIES),
        rep=random.choice(REP_NAMES),
        prospect=random.choice(PROSPECTS),
        eb=random.choice(ECONOMIC_BUYERS),
        pain=random.choice(PAIN_SCENARIOS),
        process=random.choice(PROCESS_STEPS),
        close_date=random.choice(CLOSE_DATES),
        influence=random.choice(INFLUENCE_LEVELS),
        coaching=random.choice(COACHING_STATUS),
        eb_status=random.choice(EB_STATUS),
        risk_level=random.choice(list(RISK_REASONS.keys()))
    )
    t = generate_transcript(**params)
    e = generate_extraction(**{k: v for k, v in params.items() if k != 'rep'})
    text = f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n<|user|>\n{t}<|end|>\n<|assistant|>\n{e}<|end|>'
    examples.append({'text': text, 'transcript': t, 'extraction': e})

print(f'Generated {len(examples)} training examples.')

In [ ]:
print('=== SAMPLE TRANSCRIPT ===')
print(examples[0]['transcript'])
print('\n=== EXPECTED MEDDIC-R OUTPUT ===')
print(examples[0]['extraction'])

## Step 5: Prepare Dataset for Training

In [ ]:
random.shuffle(examples)
split_idx = int(len(examples) * 0.9)
train_examples = examples[:split_idx]
test_examples = examples[split_idx:]

train_dataset = Dataset.from_list([{'text': e['text']} for e in train_examples])
test_dataset = Dataset.from_list([{'text': e['text']} for e in test_examples])

print(f'Training examples: {len(train_dataset)}')
print(f'Test examples: {len(test_dataset)}')

## Step 6: Load Phi-2 with 4-bit Quantization

We load Phi-2 in 4-bit quantization using BitsAndBytes. This reduces GPU memory requirements significantly, making training feasible on a Colab T4 GPU.

This is the PyTorch stack in action: BitsAndBytes handles quantization at the tensor level, Transformers loads the model architecture and weights, and PEFT wraps the model to inject the LoRA adapter layers.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model with 4-bit quantization...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print('Model loaded.')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 7: Configure LoRA Adapter
We configure the LoRA adapter with a rank of 16. Instead of updating all 2.7 billion parameters, we inject small low-rank matrices alongside the attention layers. Only these matrices are updated during training. The original Phi-2 weights stay frozen.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # LoRA rank
    lora_alpha=32,               # Scaling factor
    lora_dropout=0.05,           # Regularization
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    inference_mode=False
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 8: Capture Before Fine-Tuning Baseline

Before training, we run the base model on a held-out test transcript. This is the before result for comparison.

In [ ]:
def run_inference(model, tokenizer, transcript, max_new_tokens=300):
    prompt = f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n<|user|>\n{transcript}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        torch.cuda.empty_cache()
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


test_transcript = test_examples[0]['transcript']
test_expected = test_examples[0]['extraction']

print('Running base model (before fine-tuning)...\n')
before_output = run_inference(model, tokenizer, test_transcript)

print('=== TEST TRANSCRIPT ===')
print(test_transcript)
print('\n=== BASE MODEL OUTPUT (BEFORE FINE-TUNING) ===')
print(before_output)

## Step 9: Train the LoRA Adapter

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    optim='adamw_torch',
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type='cosine',
    save_steps=50,
    max_seq_length=512,
    dataset_text_field='text',
    report_to='none'
)

model.enable_input_require_grads()

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=sft_config,
    tokenizer=tokenizer,
)

print('Starting training...')
print(f'Training on {len(train_dataset)} examples for {sft_config.num_train_epochs} epochs.')
print('Expected time on T4 GPU: approximately 20-30 minutes.\n')
trainer.train()
print('\nTraining complete.')

## Step 10: Before and After Comparison

We run the same test transcript through the fine-tuned model and compare against the base model output.

In [ ]:
print('Running fine-tuned model...')
after_output = run_inference(model, tokenizer, test_transcript)

print('=' * 60)
print('BEFORE vs AFTER COMPARISON')
print('=' * 60)
print('\n--- TEST TRANSCRIPT ---')
print(test_transcript)
print('\n--- BASE MODEL OUTPUT (BEFORE FINE-TUNING) ---')
print(before_output)
print('\n--- FINE-TUNED MODEL OUTPUT (AFTER) ---')
print(after_output)
print('\n--- EXPECTED MEDDIC-R OUTPUT ---')
print(test_expected)

## Step 11: Save the LoRA Adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')
print("""
To reload this adapter in a future session:

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained('microsoft/phi-2')
model = PeftModel.from_pretrained(base_model, './apex-meddicr-adapter')
tokenizer = AutoTokenizer.from_pretrained('./apex-meddicr-adapter')
""")

## Step 12: Run Additional Test Examples

In [ ]:
print('Running fine-tuned model on 3 additional test examples...\n')
for i, example in enumerate(test_examples[1:4], 1):
    print(f'=== TEST EXAMPLE {i} ===')
    print('\nTranscript:')
    print(example['transcript'])
    output = run_inference(model, tokenizer, example['transcript'])
    print('\nFine-Tuned Model Output:')
    print(output)
    print('\nExpected Output:')
    print(example['extraction'])
    print('\n' + '-' * 60 + '\n')

---

## What We Built

1. Generated 150 synthetic sales call transcripts paired with correctly structured MEDDIC-R qualification records
2. Loaded Phi-2 in 4-bit quantization to fit within Colab T4 GPU memory constraints
3. Configured a LoRA adapter targeting the attention layers, updating less than 1% of total model parameters
4. Fine-tuned the adapter using Hugging Face TRL SFTTrainer over 3 epochs
5. Compared base model output against fine-tuned output on held-out test examples
6. Saved the adapter for reuse without retraining

The base model approximates MEDDIC structure but misses Apex-specific field definitions: it conflates Decision Criteria and Decision Process, misses engagement status on Economic Buyer, omits the Risk Score entirely, and does not apply completeness rules. The fine-tuned model internalizes these rules. The logic is no longer in the prompt. It is in the weights.

**Next steps for production:**
- Expand the dataset with real anonymized transcripts to improve robustness on edge cases
- Add a confidence scoring layer to flag extractions below a quality threshold for human review
- Retrain the adapter quarterly as your qualification schema evolves
- Consider merging the LoRA adapter into the base model weights for faster inference once stable

---

*Part of the Building with Agentic AI series. Article 8: Make Your Intelligence Layer Your MOAT: Fine-Tuning LLMs for GTM with a Full Colab Build.*